# Fixtures Analysis

Loads `fixtures` and `fixtures_enhanced` written by `fixtures_etl.py` and explores standings and scoring.

In [9]:
import duckdb, glob, pandas as pd, numpy as np, plotly.express as px

In [10]:
db_candidates = sorted(glob.glob('../../data/mydb2024-25*.duckdb'))
db_path = db_candidates[-1] if db_candidates else '../data/mydb.duckdb'
print(f'Using DuckDB database: {db_path}')
con = duckdb.connect(db_path)

Using DuckDB database: ../../data\mydb2024-25.duckdb


In [11]:
df_fixtures = con.execute('SELECT * FROM fixtures').df()
df_fixtures.head()

,fixtureId,seasonId,fixtureNumber,nameLocal,nameLatin,startTimeLocal,startTimeUTC,roundNumber,externalId,competitors,entityId_home,entityId_away,name_team_home,name_team_away,score_home,score_away,resultPlace_home,resultPlace_away,session_id
0,00c08679-4374-11ef-80bd-73cf0bc66b45,cabcf509-4373-11ef-a370-9d3c1e90234a,61,TSV Hannover-Burgdorf vs. SG Flensburg-Handewitt,<NA>,2024-10-20T15:00:00,2024-10-20T13:00:00,7,57979,[{'entityId': 'fe88f93b-3952-11ef-aa5d-af5c55c...,fea93a14-3952-11ef-a7e0-af5c55c3771d,fe88f93b-3952-11ef-aa5d-af5c55c3771d,TSV Hannover-Burgdorf,SG Flensburg-Handewitt,31,30,1,2,2658
1,00febf5a-4374-11ef-9a3c-a3f6150225cd,cabcf509-4373-11ef-a370-9d3c1e90234a,62,SC Magdeburg vs. SC DHfK Leipzig,<NA>,2024-10-20T16:00:00,2024-10-20T14:00:00,7,57980,[{'entityId': 'fe848316-3952-11ef-8185-af5c55c...,fe848316-3952-11ef-8185-af5c55c3771d,feace61d-3952-11ef-ae23-af5c55c3771d,SC Magdeburg,SC DHfK Leipzig,35,29,1,2,2653
2,0182fb4e-4374-11ef-a879-691708cc0833,cabcf509-4373-11ef-a370-9d3c1e90234a,63,TBV Lemgo Lippe vs. TVB Stuttgart,<NA>,2024-10-20T16:30:00,2024-10-20T14:30:00,7,57981,[{'entityId': 'fe99a935-3952-11ef-9dd4-af5c55c...,fe99a935-3952-11ef-9dd4-af5c55c3771d,febb3114-3952-11ef-b6f7-af5c55c3771d,TBV Lemgo Lippe,TVB Stuttgart,28,24,1,2,2654
3,037982a4-4374-11ef-982a-1f74ee999933,cabcf509-4373-11ef-a370-9d3c1e90234a,64,SC DHfK Leipzig vs. MT Melsungen,<NA>,2024-10-24T19:00:00,2024-10-24T17:00:00,8,57982,[{'entityId': 'feace61d-3952-11ef-ae23-af5c55c...,feace61d-3952-11ef-ae23-af5c55c3771d,feb41b46-3952-11ef-9df6-af5c55c3771d,SC DHfK Leipzig,MT Melsungen,27,28,2,1,2661
4,03a7d8f5-4374-11ef-abc8-75b52ec025f2,cabcf509-4373-11ef-a370-9d3c1e90234a,65,Handball Sport Verein Hamburg vs. TSV Hannover...,<NA>,2024-10-24T19:00:00,2024-10-24T17:00:00,8,57983,[{'entityId': '0045fbc4-3953-11ef-a217-af5c55c...,0045fbc4-3953-11ef-a217-af5c55c3771d,fea93a14-3952-11ef-a7e0-af5c55c3771d,Handball Sport Verein Hamburg,TSV Hannover-Burgdorf,32,32,1,1,2659


In [12]:
# Enhanced table (may not exist on first run)
try:
    df_fx_enh = con.execute('SELECT * FROM fixtures_enhanced').df()
except Exception:
    df_fx_enh = pd.DataFrame()
df_fx_enh.head() if not df_fx_enh.empty else 'fixtures_enhanced not available' 

,fixtureId,seasonId,fixtureNumber,nameLocal,nameLatin,startTimeLocal,startTimeUTC,roundNumber,externalId,entityId_home,...,standing_home,standing_away,wins_home,draws_home,losses_home,pts_against_home,wins_away,draws_away,losses_away,pts_against_away
0,ccdbc62f-4373-11ef-8506-0f9723377f57,cabcf509-4373-11ef-a370-9d3c1e90234a,1,TSV Hannover-Burgdorf vs. VfL Gummersbach,<NA>,2024-09-05T19:00:00,2024-09-05 17:00:00,1,57919,fea93a14-3952-11ef-a7e0-af5c55c3771d,...,18,1,0,0,1,2,1,0,0,0
1,cee2c85c-4373-11ef-94e2-cd77cb69a9df,cabcf509-4373-11ef-a370-9d3c1e90234a,2,TBV Lemgo Lippe vs. MT Melsungen,<NA>,2024-09-05T19:00:00,2024-09-05 17:00:00,1,57920,fe99a935-3952-11ef-9dd4-af5c55c3771d,...,18,1,0,0,1,2,1,0,0,0
2,d0f835ea-4373-11ef-a915-811a78587cbc,cabcf509-4373-11ef-a370-9d3c1e90234a,3,Rhein-Neckar Löwen vs. THW Kiel,<NA>,2024-09-05T20:30:00,2024-09-05 18:30:00,1,57921,fe80598a-3952-11ef-914c-af5c55c3771d,...,2,17,1,0,0,0,0,0,1,2
3,d1f5d36c-4373-11ef-ada8-811a78587cbc,cabcf509-4373-11ef-a370-9d3c1e90234a,4,FRISCH AUF! Göppingen vs. Handball Sport Verei...,<NA>,2024-09-06T19:00:00,2024-09-06 17:00:00,1,57922,fea4237d-3952-11ef-9fcd-af5c55c3771d,...,4,5,0,1,0,1,0,1,0,1
4,d3e126de-4373-11ef-9661-2f970b50aee5,cabcf509-4373-11ef-a370-9d3c1e90234a,5,SG Flensburg-Handewitt vs. HC Erlangen,<NA>,2024-09-06T20:00:00,2024-09-06 18:00:00,1,57923,fe88f93b-3952-11ef-aa5d-af5c55c3771d,...,1,18,1,0,0,0,0,0,1,2


In [13]:
# Parse times and scores
for col in ['score_home','score_away']:
    if col in df_fixtures.columns:
        df_fixtures[col] = pd.to_numeric(df_fixtures[col], errors='coerce')
if 'startTimeUTC' in df_fixtures.columns:
    df_fixtures['startTimeUTC'] = pd.to_datetime(df_fixtures['startTimeUTC'], errors='coerce')
df_fixtures['goals_total'] = df_fixtures.get('score_home',0) + df_fixtures.get('score_away',0)
df_fixtures[['score_home','score_away','goals_total']].describe()

,score_home,score_away,goals_total
count,306.000000,306.000000,306.000000
mean,30.013072,28.859477,58.872549
std,4.832251,4.630249,6.991805
min,18.000000,15.000000,40.000000
25%,26.000000,25.250000,54.000000
50%,30.000000,29.000000,59.000000
75%,33.000000,32.000000,63.000000
max,45.000000,43.000000,80.000000


In [14]:
# Goals distribution
px.histogram(df_fixtures, x='goals_total', nbins=30, title='Total Goals per Match')

In [15]:
# Top scoring teams (home+away)
home = df_fixtures.groupby('name_team_home')['score_home'].sum().rename('goals_home') if 'name_team_home' in df_fixtures.columns else pd.Series(dtype=float)
away = df_fixtures.groupby('name_team_away')['score_away'].sum().rename('goals_away') if 'name_team_away' in df_fixtures.columns else pd.Series(dtype=float)
df_goals = pd.concat([home, away], axis=1).fillna(0)
df_goals['goals_total'] = df_goals.sum(axis=1)
px.bar(df_goals.sort_values('goals_total', ascending=False).reset_index().head(20), x='index', y='goals_total', title='Top Teams by Goals (Total)')

In [16]:
# Standings snapshot from enhanced table (if available)
if not df_fx_enh.empty:
    cols = [c for c in ['wins_home','draws_home','losses_home','wins_away','draws_away','losses_away'] if c in df_fx_enh.columns]
    display(df_fx_enh[cols].describe())
else:
    print('fixtures_enhanced not available yet.')

,wins_home,draws_home,losses_home,wins_away,draws_away,losses_away
count,306.000000,306.000000,306.000000,306.000000,306.000000,306.000000
mean,8.238562,1.153595,8.107843,8.104575,1.147059,8.248366
std,6.577055,1.399978,6.675485,6.490511,1.405356,6.596354
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,0.000000,3.000000,3.000000,0.000000,3.000000
50%,7.000000,1.000000,6.000000,7.000000,1.000000,6.000000
75%,12.000000,2.000000,12.000000,12.000000,2.000000,12.000000
max,27.000000,5.000000,31.000000,28.000000,5.000000,30.000000
